# Training Data Preprocessing

Builds a clean, model-ready training corpus by:
1. Consolidating all 103 SDRF TSV files into one normalised table  
2. Loading and concatenating PubText JSON sections per PXD  
3. Joining SDRF labels to publication text on PXD  
4. Aligning column names to the `SampleSubmission.csv` schema  
5. Saving `outputs/train_preprocessed.csv`

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 60)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR        = ROOT / 'data'
SDRF_DIR        = DATA_DIR / 'TrainingSDRFs'
PUBTEXT_DIR     = DATA_DIR / 'TrainingPubText'
SUBMISSION_PATH = DATA_DIR / 'SampleSubmission.csv'
OUTPUT_DIR      = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('SDRF files :', len(list(SDRF_DIR.glob('PXD*.tsv'))))
print('PubText JSONs:', len(list(PUBTEXT_DIR.glob('*_PubText.json'))))

Root: c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data
SDRF files : 103
PubText JSONs: 103


## Step 1 – Consolidate SDRF TSV files
Each TSV has its own column set; we take the union and forward-fill `Not Applicable` for missing fields.

In [2]:
# Column-name normalisation map (raw SDRF header → SampleSubmission column)
COL_RENAME = {
    # biology / sample
    'SourceName':              'Source Name',
    'Organism':                'Characteristics[Organism]',
    'OrganismPart':            'Characteristics[OrganismPart]',
    'CellType':                'Characteristics[CellType]',
    'CellLine':                'Characteristics[CellLine]',
    'Disease':                 'Characteristics[Disease]',
    'DevelopmentalStage':      'Characteristics[DevelopmentalStage]',
    'Sex':                     'Characteristics[Sex]',
    'Age':                     'Characteristics[Age]',
    'AncestryCategory':        'Characteristics[AncestryCategory]',
    'BiologicalReplicate':     'Characteristics[BiologicalReplicate]',
    'TechnicalReplicate':      'Characteristics[TechnicalReplicate]',
    # instrument / method
    'Instrument':              'Comment[Instrument]',
    'Label':                   'Comment[Label]',
    'CleavageAgent':           'Comment[CleavageAgent]',
    'FragmentationMethod':     'Comment[FragmentationMethod]',
    'FractionIdentifier':      'Comment[FractionIdentifier]',
    'PrecursorMassTolerance':  'Comment[PrecursorMassTolerance]',
    'FragmentMassTolerance':   'Comment[FragmentMassTolerance]',
    # file
    'AssayName':               'Assay Name',
}

# Publication text section keys (in order of relevance)
PUBTEXT_SECTIONS = ['TITLE', 'ABSTRACT', 'INTRO', 'METHODS', 'RESULTS', 'DISCUSS', 'FIG', 'SUPPL']

print('Normalisation map ready:', len(COL_RENAME), 'entries')

Normalisation map ready: 20 entries


In [3]:
sdrf_frames = []

for tsv_path in sorted(SDRF_DIR.glob('PXD*.tsv')):
    pxd = tsv_path.stem.split('_')[0]
    try:
        df = pd.read_csv(tsv_path, sep='\t', dtype=str)
    except Exception as exc:
        print(f'  SKIP {tsv_path.name}: {exc}')
        continue

    # Normalise column names using the map; keep unmapped columns as-is
    df.rename(columns=COL_RENAME, inplace=True)

    # Handle repeated 'Modification' columns (they come in as Modification, Modification.1 …)
    mod_cols = [c for c in df.columns if c.lower().startswith('modification')]
    if mod_cols:
        df['Characteristics[Modification]'] = df[mod_cols].apply(
            lambda row: ';'.join(v for v in row.dropna() if v.strip()), axis=1
        )
        df.drop(columns=mod_cols, inplace=True)

    df.insert(0, 'PXD', pxd)
    sdrf_frames.append(df)

sdrf_raw = pd.concat(sdrf_frames, ignore_index=True, sort=False)
print(f'Consolidated SDRF: {sdrf_raw.shape[0]:,} rows × {sdrf_raw.shape[1]} columns')
print('Unique PXD datasets:', sdrf_raw['PXD'].nunique())
sdrf_raw.head(3)

Consolidated SDRF: 36,280 rows × 176 columns
Unique PXD datasets: 103


,PXD,Source Name,Characteristics[Organism],Characteristics[OrganismPart],Characteristics[CellType],Characteristics[BiologicalReplicate],Assay Name,comment[plasmodium strain],comment[proteomexchange accession number],comment[data file],comment[file uri],Comment[FractionIdentifier],Characteristics[TechnicalReplicate],Comment[Label],Comment[Instrument],Comment[FragmentationMethod],Comment[CleavageAgent],Comment[PrecursorMassTolerance],Comment[FragmentMassTolerance],Characteristics[Modification],Characteristics[AncestryCategory],Characteristics[Age],Characteristics[DevelopmentalStage],Characteristics[Sex],Characteristics[Disease],MaterialType,technology type,MS2MassAnalyzer,Age.1,Sex.1,FractionationMethod,CollisionEnergy,OrganismPart.1,Separation,Characteristics[CellLine],comment[isolation width],factor value[isolation width],Treatment,CleavageAgent.1,Treatment.1,characteristics[cultured cell],EnrichmentMethod,EnrichmentMethod.1,characteristics[individual],characteristics[mass],SpikedCompound,SpikedCompound.1,TechnicalReplicate.1,comment[file url],comment[PTM],...,characteristics[subtype],factor value[subtype],characteristics[growth condition],ReductionReagent,AlkylationReagent,comment[Proteomics Data Acquisition Method],characteristics[sample amount],characteristics[proteomexchange accession number],characteristics[CERAD],characteristics[BRAAK],characteristics[PMI..hr.],characteristics[Clinical information],comment[Batch],Time,Experiment,characteristics[plasmodium strain],SpikedCompound.2,SpikedCompound.3,SpikedCompound.4,SpikedCompound.5,SpikedCompound.6,SpikedCompound.7,SpikedCompound.8,SpikedCompound.9,SpikedCompound.10,SpikedCompound.11,SpikedCompound.12,TumorSize,characteristics[Fuhrman Nuclear Grade],characteristics[Pathologic],characteristics[region],characteristics[sample],characteristics[apoE.status],characteristics[broadman area],comment[batch],PooledSample,EnrichmentMethod.2,Treatment.2,Treatment.3,characteristics[Age at Death],characteristics[Amyloid stage],characteristics[pool],factor value[pool],Compound.1,Bait,characteristics[infect],characteristics[multiplicities of infection],characteristics[multiplicities of infection].1,Time.1,factor value[multiplicities of infection]
0,PXD000070,Sample 1,plasmodium falciparum,human erythrocytes,schizont,1,run 1,3D7,PXD000070,OTPf-IMACDT2010Mar11-02.raw,https://ftp.pride.ebi.ac.uk/pride/data/archive/2014/04/P...,1,1,label free sample,AC=MS:1001742;NT=LTQ Orbitrap Velos,ETD,AC=MS:1001251;NT=Trypsin,20 ppm,1.0005 Da,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PXD000070,Sample 2,plasmodium falciparum,human erythrocytes,schizont,2,run 2,3D7,PXD000070,OTPf-IMACDT2010Mar11-01.raw,https://ftp.pride.ebi.ac.uk/pride/data/archive/2014/04/P...,2,1,label free sample,AC=MS:1001742;NT=LTQ Orbitrap Velos,ETD,AC=MS:1001251;NT=Trypsin,20 ppm,1.0005 Da,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PXD000070,Sample 3,plasmodium falciparum,human erythrocytes,schizont,3,run 3,3D7,PXD000070,OTPf-IMACDT2010Mar10-01.raw,https://ftp.pride.ebi.ac.uk/pride/data/archive/2014/04/P...,3,1,label free sample,AC=MS:1001742;NT=LTQ Orbitrap Velos,ETD,AC=MS:1001251;NT=Trypsin,20 ppm,1.0005 Da,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

## Step 2 – Align to `SampleSubmission.csv` schema
We keep only the columns that exist in the submission template (the target label space) and fill anything missing with `Not Applicable`.

In [4]:
submission_cols = pd.read_csv(SUBMISSION_PATH, nrows=0).columns.tolist()
# Columns that are label targets (exclude ID, PXD, Raw Data File)
label_cols = [c for c in submission_cols if c not in ('ID', 'PXD', 'Raw Data File')]

print(f'Submission has {len(submission_cols)} columns → {len(label_cols)} label targets')

# Add any missing label columns with 'Not Applicable'
for col in label_cols:
    if col not in sdrf_raw.columns:
        sdrf_raw[col] = 'Not Applicable'

# Fill NaN → 'Not Applicable'
sdrf_raw[label_cols] = sdrf_raw[label_cols].fillna('Not Applicable')

# Keep only the columns we care about
keep_cols = ['PXD'] + [c for c in submission_cols if c in sdrf_raw.columns]
sdrf_aligned = sdrf_raw[keep_cols].copy()

print(f'Aligned SDRF: {sdrf_aligned.shape}')
sdrf_aligned.head(3)

Submission has 81 columns → 78 label targets
Aligned SDRF: (36280, 80)


,PXD,PXD,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],Characteristics[CellLine],Characteristics[CellPart],Characteristics[CellType],Characteristics[CleavageAgent],Characteristics[Compound],Characteristics[ConcentrationOfCompound],Characteristics[Depletion],Characteristics[DevelopmentalStage],Characteristics[DiseaseTreatment],Characteristics[Disease],Characteristics[GeneticModification],Characteristics[Genotype],Characteristics[GrowthRate],Characteristics[Label],Characteristics[MaterialType],Characteristics[Modification],Characteristics[Modification].1,Characteristics[Modification].2,Characteristics[Modification].3,Characteristics[Modification].4,Characteristics[Modification].5,Characteristics[Modification].6,Characteristics[NumberOfBiologicalReplicates],Characteristics[NumberOfSamples],Characteristics[NumberOfTechnicalReplicates],Characteristics[OrganismPart],Characteristics[Organism],Characteristics[OriginSiteDisease],Characteristics[PooledSample],Characteristics[ReductionReagent],Characteristics[SamplingTime],Characteristics[Sex],Characteristics[Specimen],Characteristics[SpikedCompound],Characteristics[Staining],Characteristics[Strain],Characteristics[SyntheticPeptide],Characteristics[Temperature],Characteristics[Time],Characteristics[Treatment],Characteristics[TumorCellularity],Characteristics[TumorGrade],Characteristics[TumorSite],Characteristics[TumorSize],Characteristics[TumorStage],Comment[AcquisitionMethod],Comment[CollisionEnergy],Comment[EnrichmentMethod],Comment[FlowRateChromatogram],Comment[FractionIdentifier],Comment[FractionationMethod],Comment[FragmentMassTolerance],Comment[FragmentationMethod],Comment[GradientTime],Comment[Instrument],Comment[IonizationType],Comment[MS2MassAnalyzer],Comment[NumberOfFractions],Comment[NumberOfMissedCleavages],Comment[PrecursorMassTolerance],Comment[Separation],FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,PXD000070,PXD000070,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,Not Applicable,Not Applicable,schizont,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,human erythrocytes,plasmodium falciparum,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,Not Applicable,1.0005 Da,ETD,Not Applicable,AC=MS:1001742;NT=LTQ Orbitrap Velos,Not Applicable,Not Applicable,Not Applicable,Not Applicable,20 ppm,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,PXD000070,PXD000070,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,2,Not Applicable,Not Applicable,schizont,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,human erythrocytes

## Step 3 – Load Publication Text
Concatenate all text sections per PXD into a single string.

In [5]:
def load_pubtext(folder: Path) -> pd.DataFrame:
    records = []
    for jp in sorted(folder.glob('*_PubText.json')):
        pxd = jp.stem.replace('_PubText', '')
        try:
            payload = json.loads(jp.read_text(encoding='utf-8'))
        except Exception:
            continue

        if isinstance(payload, dict):
            parts = []
            for sec in PUBTEXT_SECTIONS:
                val = payload.get(sec, '')
                if val and isinstance(val, str):
                    parts.append(val.strip())
            # Fallback: any remaining string values
            if not parts:
                parts = [str(v) for v in payload.values() if isinstance(v, str) and v.strip()]
            text = ' '.join(parts)
        elif isinstance(payload, list):
            text = ' '.join(str(item) for item in payload)
        else:
            text = str(payload)

        records.append({'PXD': pxd, 'pub_text': text.strip()})
    return pd.DataFrame(records)


pubtext_df = load_pubtext(PUBTEXT_DIR)
print(f'Loaded {len(pubtext_df)} PubText documents')
pubtext_df['text_len'] = pubtext_df['pub_text'].str.split().str.len()
print(pubtext_df['text_len'].describe().to_string())
pubtext_df.head(3)

Loaded 103 PubText documents
count      103.000000
mean      6708.524272
std       2330.833487
min       1097.000000
25%       5251.000000
50%       6248.000000
75%       8069.000000
max      13775.000000


,PXD,pub_text,text_len
0,PXD000070,Confident and sensitive phosphoproteomics using combinat...,7664
1,PXD000534,Identification of Novel Biomarker Candidates for the Imm...,6200
2,PXD000561,Functional Annotation of Proteome Encoded by Human Chrom...,4331


## Step 4 – Join SDRF labels ↔ PubText on PXD

In [7]:
# Drop any duplicate PXD column that may arise from the alignment step
sdrf_aligned = sdrf_aligned.loc[:, ~sdrf_aligned.columns.duplicated()]

train_df = sdrf_aligned.merge(pubtext_df[['PXD', 'pub_text']], on='PXD', how='inner')

print(f'Joined dataset: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'PXDs with both SDRF and PubText: {train_df["PXD"].nunique()}')

# Move pub_text to be the second column for readability
cols = ['PXD', 'pub_text'] + [c for c in train_df.columns if c not in ('PXD', 'pub_text')]
train_df = train_df[cols]

train_df.head(3)

Joined dataset: 36,280 rows × 80 columns
PXDs with both SDRF and PubText: 103


,PXD,pub_text,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],Characteristics[CellLine],Characteristics[CellPart],Characteristics[CellType],Characteristics[CleavageAgent],Characteristics[Compound],Characteristics[ConcentrationOfCompound],Characteristics[Depletion],Characteristics[DevelopmentalStage],Characteristics[DiseaseTreatment],Characteristics[Disease],Characteristics[GeneticModification],Characteristics[Genotype],Characteristics[GrowthRate],Characteristics[Label],Characteristics[MaterialType],Characteristics[Modification],Characteristics[Modification].1,Characteristics[Modification].2,Characteristics[Modification].3,Characteristics[Modification].4,Characteristics[Modification].5,Characteristics[Modification].6,Characteristics[NumberOfBiologicalReplicates],Characteristics[NumberOfSamples],Characteristics[NumberOfTechnicalReplicates],Characteristics[OrganismPart],Characteristics[Organism],Characteristics[OriginSiteDisease],Characteristics[PooledSample],Characteristics[ReductionReagent],Characteristics[SamplingTime],Characteristics[Sex],Characteristics[Specimen],Characteristics[SpikedCompound],Characteristics[Staining],Characteristics[Strain],Characteristics[SyntheticPeptide],Characteristics[Temperature],Characteristics[Time],Characteristics[Treatment],Characteristics[TumorCellularity],Characteristics[TumorGrade],Characteristics[TumorSite],Characteristics[TumorSize],Characteristics[TumorStage],Comment[AcquisitionMethod],Comment[CollisionEnergy],Comment[EnrichmentMethod],Comment[FlowRateChromatogram],Comment[FractionIdentifier],Comment[FractionationMethod],Comment[FragmentMassTolerance],Comment[FragmentationMethod],Comment[GradientTime],Comment[Instrument],Comment[IonizationType],Comment[MS2MassAnalyzer],Comment[NumberOfFractions],Comment[NumberOfMissedCleavages],Comment[PrecursorMassTolerance],Comment[Separation],FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,PXD000070,Confident and sensitive phosphoproteomics using combinat...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,Not Applicable,Not Applicable,schizont,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,human erythrocytes,plasmodium falciparum,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,Not Applicable,1.0005 Da,ETD,Not Applicable,AC=MS:1001742;NT=LTQ Orbitrap Velos,Not Applicable,Not Applicable,Not Applicable,Not Applicable,20 ppm,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,PXD000070,Confident and sensitive phosphoproteomics using combinat...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,2,Not Applicable,Not Applicable,schizont,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,Not Applicable,Not Applicable,Not Applicable,Not

## Step 5 – Quality check

In [8]:
# Missing-value summary across label columns
not_applicable_pct = (train_df[label_cols]
    .apply(lambda s: (s == 'Not Applicable').mean() * 100)
    .sort_values(ascending=False))

print('Top 15 most-sparse label columns (% Not Applicable):')
print(not_applicable_pct.head(15).to_string())
print()

# Columns that are ALWAYS 'Not Applicable' → useless for training
always_na = not_applicable_pct[not_applicable_pct == 100].index.tolist()
print(f'Columns always "Not Applicable" (will be dropped from labels): {len(always_na)}')
print(always_na[:20])

Top 15 most-sparse label columns (% Not Applicable):
Characteristics[AlkylationReagent]          100.0
Characteristics[AnatomicSiteTumor]          100.0
Characteristics[Bait]                       100.0
Characteristics[BMI]                        100.0
Characteristics[ConcentrationOfCompound]    100.0
Characteristics[CellPart]                   100.0
Characteristics[Depletion]                  100.0
Characteristics[CleavageAgent]              100.0
Characteristics[Compound]                   100.0
Characteristics[Label]                      100.0
Characteristics[Genotype]                   100.0
Characteristics[GrowthRate]                 100.0
Characteristics[GeneticModification]        100.0
Characteristics[DiseaseTreatment]           100.0
FactorValue[GeneticModification]            100.0

Columns always "Not Applicable" (will be dropped from labels): 62
['Characteristics[AlkylationReagent]', 'Characteristics[AnatomicSiteTumor]', 'Characteristics[Bait]', 'Characteristics[BMI]', 'Cha

In [9]:
# Drop columns that are always 'Not Applicable'
useful_label_cols = [c for c in label_cols if c not in always_na]
print(f'Useful label columns: {len(useful_label_cols)} / {len(label_cols)}')

# Value diversity per column
diversity = (train_df[useful_label_cols]
    .apply(lambda s: s[s != 'Not Applicable'].nunique())
    .sort_values(ascending=False))

print('\nTop 20 most diverse label columns (unique non-NA values):')
print(diversity.head(20).to_string())

Useful label columns: 16 / 78

Top 20 most diverse label columns (unique non-NA values):
Characteristics[Modification]           3357
Comment[Instrument]                      339
Characteristics[Age]                     157
Characteristics[OrganismPart]            134
Characteristics[Disease]                 127
Comment[FractionIdentifier]              117
Characteristics[CellLine]                 90
Characteristics[BiologicalReplicate]      43
Characteristics[CellType]                 30
Comment[FragmentMassTolerance]            25
Characteristics[Organism]                 19
Comment[FragmentationMethod]              17
Comment[PrecursorMassTolerance]           13
Characteristics[AncestryCategory]         11
Characteristics[DevelopmentalStage]        8
Characteristics[Sex]                       8


## Step 6 – Save preprocessed training data

In [10]:
# Final dataset: keep PXD, pub_text, and useful label columns only
final_cols = ['PXD', 'pub_text'] + useful_label_cols
# Also keep Raw Data File if present (useful for matching at inference time)
if 'Raw Data File' in train_df.columns and 'Raw Data File' not in final_cols:
    final_cols.insert(2, 'Raw Data File')

train_final = train_df[final_cols].copy()
train_final.insert(0, 'row_id', range(len(train_final)))

out_path = OUTPUT_DIR / 'train_preprocessed.csv'
train_final.to_csv(out_path, index=False)

print(f'Saved to: {out_path}')
print(f'Shape   : {train_final.shape}')
print(f'Columns : {train_final.shape[1]}')
train_final.head(3)

Saved to: c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\train_preprocessed.csv
Shape   : (36280, 19)
Columns : 19


,row_id,PXD,pub_text,Characteristics[Age],Characteristics[AncestryCategory],Characteristics[BiologicalReplicate],Characteristics[CellLine],Characteristics[CellType],Characteristics[DevelopmentalStage],Characteristics[Disease],Characteristics[Modification],Characteristics[OrganismPart],Characteristics[Organism],Characteristics[Sex],Comment[FractionIdentifier],Comment[FragmentMassTolerance],Comment[FragmentationMethod],Comment[Instrument],Comment[PrecursorMassTolerance]
0,0,PXD000070,Confident and sensitive phosphoproteomics using combinat...,Not Applicable,Not Applicable,1,Not Applicable,schizont,Not Applicable,Not Applicable,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,human erythrocytes,plasmodium falciparum,Not Applicable,1,1.0005 Da,ETD,AC=MS:1001742;NT=LTQ Orbitrap Velos,20 ppm
1,1,PXD000070,Confident and sensitive phosphoproteomics using combinat...,Not Applicable,Not Applicable,2,Not Applicable,schizont,Not Applicable,Not Applicable,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,human erythrocytes,plasmodium falciparum,Not Applicable,2,1.0005 Da,ETD,AC=MS:1001742;NT=LTQ Orbitrap Velos,20 ppm
2,2,PXD000070,Confident and sensitive phosphoproteomics using combinat...,Not Applicable,Not Applicable,3,Not Applicable,schizont,Not Applicable,Not Applicable,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;NT=Oxidatio...,human erythrocytes,plasmodium falciparum,Not Applicable,3,1.0005 Da,ETD,AC=MS:1001742;NT=LTQ Orbitrap Velos,20 ppm


## Step 7 – Per-PXD summary (sanity check)

In [11]:
per_pxd = (
    train_final
    .groupby('PXD')
    .agg(
        rows=('row_id', 'count'),
        text_tokens=('pub_text', lambda s: s.iloc[0].split().__len__()),
    )
    .sort_values('rows', ascending=False)
)
print(f'Per-PXD summary ({len(per_pxd)} datasets):')
print(per_pxd.to_string())

Per-PXD summary (103 datasets):
           rows  text_tokens
PXD                         
PXD010429  4176        11993
PXD017201  3600         6233
PXD011967  2316         7278
PXD000561  2212         4331
PXD010154  1800        10194
PXD007160  1687         4650
PXD005354  1560        10583
PXD002870  1477         5996
PXD004732  1460         1764
PXD006003  1346         8465
PXD004242  1290         8924
PXD008840  1016         9332
PXD016002  1008         6091
PXD017035   720         6253
PXD006675   594         8532
PXD009602   528         5157
PXD011799   480         8965
PXD014557   400         7185
PXD006233   378         5003
PXD013923   372         6248
PXD012131   326         5487
PXD012143   320        10832
PXD010603   311         5487
PXD005207   258         9721
PXD008722   252         6805
PXD012203   240         5092
PXD018357   240         6449
PXD002266   216         8938
PXD008440   200         6397
PXD002395   198         5841
PXD013523   198         6720
PXD005445  